# Avaliação dos modelos e do ensemble ponderado

Este notebook carrega os modelos treinados em `artifacts/models`, usa junho a agosto de 2019 como validação para calcular pesos e thresholds e preserva setembro a dezembro de 2019 como teste final.

As saídas são salvas automaticamente em:

- `reports/metrics/modeling`: métricas e dados dos gráficos em CSV;
- `reports/figures/modeling/evaluation`: gráficos em PNG (200 dpi).

O ensemble usa a mesma regra do projeto original: cada score é ponderado pelo recall do modelo. A classe positiva é **1 = dengue confirmada**.

## 1. Configuração

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dengue_pipeline").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline import DengueDataCleaner
from dengue_pipeline.paths import MODELS_DIR, MODEL_FIGURES_DIR

METRICS_DIR = PROJECT_ROOT / "reports" / "metrics" / "modeling"
EVALUATION_FIGURES_DIR = MODEL_FIGURES_DIR / "evaluation"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    "MLP": "mlp.joblib",
    "XGBoost": "xgboost.joblib",
    "LightGBM": "lightgbm.joblib",
}
MODEL_IDS = {
    "MLP": "mlp",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm",
}
MODEL_ORDER = [*MODEL_FILES, "Ensemble"]

MODEL_COLORS = {
    "MLP": "#2563eb",
    "XGBoost": "#0f766e",
    "LightGBM": "#7c3aed",
    "Ensemble": "#dc2626",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 10,
    "font.size": 10,
})

print(f"Modelos: {MODELS_DIR}")
print(f"Tabelas: {METRICS_DIR}")
print(f"Figuras: {EVALUATION_FIGURES_DIR}")

## 2. Reconstrução da validação e do teste

A validação temporal define pesos e thresholds. O teste final não participa de nenhuma escolha e serve apenas para medir o resultado pronto.

In [ ]:
TARGET = "final_classification"
ADMINISTRATIVE_FEATURES = [
    "notification_year",
    "notification_month",
    "symptom_epi_year",
    "notif_municipality",
    "notif_health_region",
    "health_facility",
]

cleaner = DengueDataCleaner()
df = cleaner.transformar_ml()

validation_mask = (df["notification_year"] == 2019) & df["notification_month"].between(6, 8)
test_mask = (df["notification_year"] == 2019) & df["notification_month"].between(9, 12)
df_validation = df[validation_mask].copy()
df_test = df[test_mask].copy()

def preparar_features(dataframe):
    target = dataframe[TARGET].astype(int)
    features = dataframe.drop(columns=[TARGET]).select_dtypes(include=["number"])
    features = features.drop(columns=ADMINISTRATIVE_FEATURES, errors="ignore")
    return features.astype("float32"), target

X_validation, y_validation = preparar_features(df_validation)
X_test, y_test = preparar_features(df_test)

test_summary = pd.DataFrame([
    {"split": "validation", "period_start": "2019-06", "period_end": "2019-08", "n_samples": len(y_validation), "n_features": X_validation.shape[1], "n_confirmed": int(y_validation.sum()), "n_discarded": int((y_validation == 0).sum()), "positive_rate": float(y_validation.mean())},
    {"split": "test", "period_start": "2019-09", "period_end": "2019-12", "n_samples": len(y_test), "n_features": X_test.shape[1], "n_confirmed": int(y_test.sum()), "n_discarded": int((y_test == 0).sum()), "positive_rate": float(y_test.mean())},
])
test_summary.to_csv(METRICS_DIR / "test_set_summary.csv", index=False, encoding="utf-8-sig")

print(f"Validação: {X_validation.shape[0]:,} registros ({y_validation.mean():.2%} confirmados)")
print(f"Teste final: {X_test.shape[0]:,} registros ({y_test.mean():.2%} confirmados)")

# A base completa é grande, então ela não precisa continuar na memória durante as previsões.
del cleaner, df, df_validation, df_test

test_summary

## 3. Carregamento e avaliação

Os thresholds são escolhidos na validação pelo maior **F1-score**, equilibrando precisão e recall. Depois, o recall de cada modelo no threshold escolhido é transformado em peso para a média ponderada. Os valores finais ficam anotados no resultado desta seção e são copiados como constantes para `api.py`.

- **accuracy**: proporção total de acertos;
- **balanced_accuracy**: média do recall das duas classes;
- **precision**: entre os casos previstos como dengue, quantos foram confirmados;
- **recall / sensitivity**: entre os casos confirmados, quantos foram detectados;
- **specificity**: entre os descartados, quantos foram corretamente descartados;
- **f1**: média harmônica de precisão e recall;
- **roc_auc** e **pr_auc**: discriminação usando todos os limiares.

In [ ]:
def pegar_probabilidade_positiva(modelo, dados):
    probabilidades = np.asarray(modelo.predict_proba(dados))
    return probabilidades[:, 1] if probabilidades.ndim == 2 else probabilidades


def escolher_pontos(total, limite=1000):
    return np.linspace(0, total - 1, limite).astype(int) if total > limite else np.arange(total)


def metricas_por_threshold(model_name, target, scores, thresholds):
    rows = []
    for threshold in thresholds:
        predictions = (scores >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(target, predictions, labels=[0, 1]).ravel()
        rows.append({
            "model": model_name,
            "split": "validation",
            "threshold": float(threshold),
            "accuracy": accuracy_score(target, predictions),
            "balanced_accuracy": balanced_accuracy_score(target, predictions),
            "precision": precision_score(target, predictions, zero_division=0),
            "recall": recall_score(target, predictions, zero_division=0),
            "specificity": tn / (tn + fp),
            "f1": f1_score(target, predictions, zero_division=0),
            "predicted_positive_rate": predictions.mean(),
        })
    return pd.DataFrame(rows)


def escolher_threshold(metrics):
    selected = metrics.sort_values(["f1", "recall"], ascending=False).iloc[0]
    return selected, "max_f1"


def normalizar_pesos_por_recall(recalls):
    total = sum(recalls.values())
    return {name: recall / total for name, recall in recalls.items()}


def media_ponderada(scores, weights):
    return sum(np.asarray(scores[name]) * weights[name] for name in weights)


def registrar_resultado_teste(model_name, target, scores, threshold, weight):
    predictions = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(target, predictions, labels=[0, 1]).ravel()
    metric_rows.append({
        "model": model_name, "threshold": threshold, "weight": weight,
        "n_test": len(target), "positive_rate": target.mean(),
        "predicted_positive_rate": predictions.mean(),
        "accuracy": accuracy_score(target, predictions),
        "balanced_accuracy": balanced_accuracy_score(target, predictions),
        "precision": precision_score(target, predictions, zero_division=0),
        "recall": recall_score(target, predictions, zero_division=0),
        "specificity": tn / (tn + fp),
        "f1": f1_score(target, predictions, zero_division=0),
        "roc_auc": roc_auc_score(target, scores),
        "pr_auc": average_precision_score(target, scores),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    })

    report = pd.DataFrame(classification_report(target, predictions, target_names=["Descartado (0)", "Confirmado (1)"], output_dict=True, zero_division=0)).T.reset_index().rename(columns={"index": "class"})
    report.insert(0, "model", model_name)
    report_frames.append(report)

    for actual, predicted, count in [(0, 0, tn), (0, 1, fp), (1, 0, fn), (1, 1, tp)]:
        confusion_rows.append({"model": model_name, "actual": actual, "predicted": predicted, "count": int(count)})

    fpr, tpr, roc_thresholds = roc_curve(target, scores)
    roc_points = escolher_pontos(len(fpr))
    roc_frames.append(pd.DataFrame({"model": model_name, "false_positive_rate": fpr[roc_points], "true_positive_rate": tpr[roc_points], "threshold": roc_thresholds[roc_points]}))

    pr_precision, pr_recall, pr_thresholds = precision_recall_curve(target, scores)
    pr_points = escolher_pontos(len(pr_recall))
    pr_frames.append(pd.DataFrame({"model": model_name, "recall": pr_recall[pr_points], "precision": pr_precision[pr_points], "threshold": np.append(pr_thresholds, np.nan)[pr_points]}))
    evaluation_results[model_name] = {"confusion_matrix": np.array([[tn, fp], [fn, tp]])}


THRESHOLDS_TO_TEST = np.round(np.arange(0.01, 0.951, 0.01), 2)

validation_scores = {}
test_scores = {}
model_recalls = {}
model_thresholds = {}
threshold_frames = []
selection_rows = []

for model_name, filename in MODEL_FILES.items():
    print(f"Gerando scores de {model_name}...")
    model = joblib.load(MODELS_DIR / filename)
    features = model.feature_names if getattr(model, "feature_names", None) else list(model.feature_names_in_)
    validation_scores[MODEL_IDS[model_name]] = pegar_probabilidade_positiva(model, X_validation[features])
    test_scores[MODEL_IDS[model_name]] = pegar_probabilidade_positiva(model, X_test[features])

    threshold_metrics = metricas_por_threshold(model_name, y_validation, validation_scores[MODEL_IDS[model_name]], THRESHOLDS_TO_TEST)
    selected, rule = escolher_threshold(threshold_metrics)
    threshold_metrics["selected"] = np.isclose(threshold_metrics["threshold"], selected["threshold"])
    threshold_frames.append(threshold_metrics)

    model_id = MODEL_IDS[model_name]
    model_thresholds[model_id] = float(selected["threshold"])
    model_recalls[model_id] = float(selected["recall"])
    selection_rows.append({"model": model_name, "rule": rule, **selected.to_dict()})

weights = normalizar_pesos_por_recall(model_recalls)
ensemble_validation_scores = media_ponderada(validation_scores, weights)
ensemble_test_scores = media_ponderada(test_scores, weights)

ensemble_threshold_metrics = metricas_por_threshold("Ensemble", y_validation, ensemble_validation_scores, THRESHOLDS_TO_TEST)
ensemble_selected, ensemble_rule = escolher_threshold(ensemble_threshold_metrics)
ensemble_threshold_metrics["selected"] = np.isclose(ensemble_threshold_metrics["threshold"], ensemble_selected["threshold"])
threshold_frames.append(ensemble_threshold_metrics)
ensemble_threshold = float(ensemble_selected["threshold"])
selection_rows.append({"model": "Ensemble", "rule": ensemble_rule, **ensemble_selected.to_dict()})

print("Valores para manter como constantes em api.py:")
print(f"ENSEMBLE_WEIGHTS = {weights}")
print(f"ENSEMBLE_THRESHOLD = {ensemble_threshold:.2f}")
display(pd.DataFrame(selection_rows)[["model", "threshold", "precision", "recall", "f1", "rule"]])
display(pd.DataFrame({"model": list(weights), "weight": list(weights.values()), "validation_recall": [model_recalls[name] for name in weights]}))

metric_rows = []
report_frames = []
confusion_rows = []
roc_frames = []
pr_frames = []
evaluation_results = {}

for model_name in MODEL_FILES:
    model_id = MODEL_IDS[model_name]
    registrar_resultado_teste(model_name, y_test, test_scores[model_id], model_thresholds[model_id], weights[model_id])
registrar_resultado_teste("Ensemble", y_test, ensemble_test_scores, ensemble_threshold, 1.0)

SELECTED_THRESHOLDS = {model_name: model_thresholds[MODEL_IDS[model_name]] for model_name in MODEL_FILES}
SELECTED_THRESHOLDS["Ensemble"] = ensemble_threshold

metrics_df = pd.DataFrame(metric_rows).sort_values("pr_auc", ascending=False)
classification_report_df = pd.concat(report_frames, ignore_index=True)
confusion_matrices_df = pd.DataFrame(confusion_rows)
roc_curves_df = pd.concat(roc_frames, ignore_index=True)
precision_recall_curves_df = pd.concat(pr_frames, ignore_index=True)
threshold_metrics_df = pd.concat(threshold_frames, ignore_index=True)
selected_thresholds_df = pd.DataFrame(selection_rows)

tables = {
    "model_metrics.csv": metrics_df,
    "classification_report.csv": classification_report_df,
    "confusion_matrices.csv": confusion_matrices_df,
    "roc_curve_points.csv": roc_curves_df,
    "precision_recall_curve_points.csv": precision_recall_curves_df,
    "threshold_metrics.csv": threshold_metrics_df,
    "selected_thresholds.csv": selected_thresholds_df,
}
for filename, table in tables.items():
    table.to_csv(METRICS_DIR / filename, index=False, encoding="utf-8-sig")

display_columns = ["model", "threshold", "weight", "accuracy", "balanced_accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
metrics_df[display_columns].round(4)

## 4. Precisão e recall dos modelos por threshold

Comparação direta dos três modelos usando o conjunto de validação.

In [ ]:
THRESHOLDS_RESUMO = [0.03, 0.05, 0.07, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

comparacao_thresholds = threshold_metrics_df[
    threshold_metrics_df["model"].isin(MODEL_FILES)
    & threshold_metrics_df["threshold"].isin(THRESHOLDS_RESUMO)
][["model", "threshold", "precision", "recall", "f1"]].copy()

comparacao_thresholds[["precision", "recall", "f1"]] *= 100
comparacao_thresholds = comparacao_thresholds.rename(columns={
    "model": "Modelo",
    "threshold": "Threshold",
    "precision": "Precisão (%)",
    "recall": "Recall (%)",
    "f1": "F1 (%)",
})

display(comparacao_thresholds.round(2).reset_index(drop=True))

## 5. Efeito do threshold no ensemble ponderado

O threshold não altera o score ponderado calculado. Ele define a partir de qual score o paciente será classificado como positivo, alterando precisão, recall e quantidade de previsões positivas.

In [ ]:
impacto_threshold_ensemble = threshold_metrics_df[
    (threshold_metrics_df["model"] == "Ensemble")
    & threshold_metrics_df["threshold"].isin(THRESHOLDS_RESUMO)
][["threshold", "precision", "recall", "f1", "accuracy", "predicted_positive_rate"]].copy()

impacto_threshold_ensemble[[
    "precision", "recall", "f1", "accuracy", "predicted_positive_rate"
]] *= 100
impacto_threshold_ensemble = impacto_threshold_ensemble.rename(columns={
    "threshold": "Threshold",
    "precision": "Precisão (%)",
    "recall": "Recall (%)",
    "f1": "F1 (%)",
    "accuracy": "Acurácia (%)",
    "predicted_positive_rate": "Previstos positivos (%)",
})

print(f"Score ponderado médio na validação: {ensemble_validation_scores.mean():.2%}")
display(impacto_threshold_ensemble.round(2).reset_index(drop=True))

## 6. Comparação das principais métricas

In [ ]:
metric_labels = {
    "accuracy": "Acurácia",
    "balanced_accuracy": "Acurácia\nbalanceada",
    "precision": "Precisão",
    "recall": "Recall",
    "f1": "F1-score",
    "roc_auc": "ROC AUC",
    "pr_auc": "PR AUC",
}

plot_data = metrics_df.set_index("model")[list(metric_labels)].rename(
    columns=metric_labels
)

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(plot_data.columns))
bar_width = 0.24
center = (len(MODEL_ORDER) - 1) / 2

for index, model_name in enumerate(MODEL_ORDER):
    values = plot_data.loc[model_name].values
    bars = ax.bar(
        x + (index - center) * bar_width,
        values,
        width=bar_width,
        label=model_name,
        color=MODEL_COLORS[model_name],
        alpha=0.9,
    )
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.012,
            f"{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#334155",
            rotation=90,
        )

ax.set_title("Comparação das Principais Métricas no Conjunto de Teste", pad=16)
ax.set_xticks(x)
ax.set_xticklabels(plot_data.columns)
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("Resultado")
ax.grid(axis="y", linestyle="--", alpha=0.22)
ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=4, loc="upper center")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="both", length=0)

fig.tight_layout()
fig.savefig(
    EVALUATION_FIGURES_DIR / "model_metrics_comparison.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 7. Curvas ROC e Precision–Recall

A curva ROC compara sensibilidade e taxa de falsos positivos. A curva Precision–Recall dá mais visibilidade ao desempenho sobre a classe positiva e costuma ser mais informativa quando há desbalanceamento.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for model_name in MODEL_ORDER:
    curve = roc_curves_df[roc_curves_df["model"] == model_name]
    auc_value = metrics_df.set_index("model").loc[model_name, "roc_auc"]
    ax.plot(
        curve["false_positive_rate"],
        curve["true_positive_rate"],
        color=MODEL_COLORS[model_name],
        linewidth=2.2,
        label=f"{model_name} (AUC = {auc_value:.3f})",
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="#94a3b8", linewidth=1.3, label="Aleatório")
ax.set_title("Curvas ROC dos Modelos", pad=14)
ax.set_xlabel("Taxa de falsos positivos (1 - especificidade)")
ax.set_ylabel("Taxa de verdadeiros positivos (recall)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(linestyle="--", alpha=0.22)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower right")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="both", length=0)
fig.tight_layout()
fig.savefig(EVALUATION_FIGURES_DIR / "roc_curves.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


fig, ax = plt.subplots(figsize=(9, 7))
for model_name in MODEL_ORDER:
    curve = precision_recall_curves_df[
        precision_recall_curves_df["model"] == model_name
    ]
    auc_value = metrics_df.set_index("model").loc[model_name, "pr_auc"]
    ax.plot(
        curve["recall"],
        curve["precision"],
        color=MODEL_COLORS[model_name],
        linewidth=2.2,
        label=f"{model_name} (AP = {auc_value:.3f})",
    )

prevalence = y_test.mean()
ax.axhline(
    prevalence,
    linestyle="--",
    color="#94a3b8",
    linewidth=1.3,
    label=f"Baseline ({prevalence:.1%})",
)
ax.set_title("Curvas Precision–Recall dos Modelos", pad=14)
ax.set_xlabel("Recall")
ax.set_ylabel("Precisão")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(linestyle="--", alpha=0.22)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower left")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="both", length=0)
fig.tight_layout()
fig.savefig(
    EVALUATION_FIGURES_DIR / "precision_recall_curves.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 8. Matrizes de confusão

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, model_name in zip(axes.flat, MODEL_ORDER):
    matrix = evaluation_results[model_name]["confusion_matrix"]
    row_totals = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(
        matrix,
        row_totals,
        out=np.zeros_like(matrix, dtype=float),
        where=row_totals != 0,
    )
    image = ax.imshow(normalized, cmap="Blues", vmin=0, vmax=1)

    for row in range(2):
        for column in range(2):
            text_color = "white" if normalized[row, column] > 0.52 else "#0f172a"
            ax.text(
                column,
                row,
                f"{matrix[row, column]:,}".replace(",", ".")
                + f"\n({normalized[row, column]:.1%})",
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color=text_color,
            )

    ax.set_title(model_name, fontsize=12, pad=10)
    ax.set_xticks([0, 1], labels=["Descartado", "Confirmado"])
    ax.set_yticks([0, 1], labels=["Descartado", "Confirmado"])
    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe real")
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle("Matrizes de Confusão no Conjunto de Teste", fontsize=16, fontweight="bold", y=1.01)
fig.colorbar(image, ax=axes, fraction=0.025, pad=0.03, label="Proporção dentro da classe real")
fig.savefig(
    EVALUATION_FIGURES_DIR / "confusion_matrices.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 9. Sensibilidade ao limiar de decisão

As curvas abaixo usam somente a validação. A linha vertical marca o threshold selecionado para cada modelo e para o ensemble.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)

for index, (ax, model_name) in enumerate(zip(axes.flat, MODEL_ORDER)):
    data = threshold_metrics_df[threshold_metrics_df["model"] == model_name]
    ax.plot(data["threshold"], data["precision"], label="Precisão", color="#2563eb", linewidth=3)
    ax.plot(data["threshold"], data["recall"], label="Recall", color="#dc2626", linewidth=3)
    ax.plot(data["threshold"], data["f1"], label="F1-score", color="#0f766e", linewidth=3)
    ax.axvline(SELECTED_THRESHOLDS[model_name], color="#94a3b8", linestyle="--", linewidth=1.2)
    ax.set_title(model_name, fontsize=12, pad=10)
    ax.set_xlim(0.01, 0.95)
    ax.set_ylim(0, 1.02)
    if index >= 2:
        ax.set_xlabel("Limiar")
    if index % 2 == 0:
        ax.set_ylabel("Resultado")
    ax.grid(linestyle="--", alpha=0.22)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.01))
fig.suptitle("Precisão, Recall e F1 por Limiar de Decisão", fontsize=16, fontweight="bold", y=1.06)
fig.tight_layout()
fig.savefig(
    EVALUATION_FIGURES_DIR / "threshold_analysis.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 10. Arquivos gerados

In [ ]:
generated_files = sorted(METRICS_DIR.glob("*.csv")) + sorted(EVALUATION_FIGURES_DIR.glob("*.png"))
pd.DataFrame({
    "arquivo": [str(path.relative_to(PROJECT_ROOT)) for path in generated_files],
    "tamanho_kb": [round(path.stat().st_size / 1024, 1) for path in generated_files],
})